# M1 dev trials on free Colab

35 draft dev tasks x 4 independent trials on Groq, then a two-run flip check. Never runs the holdout. CPU runtime only, no paid compute. Keys belong in Colab Secrets, never in cells or outputs.

Add GROQ_API_KEY and GEMINI_API_KEY in the key icon panel and enable notebook access. Stop on HTTP 403/1010; do not change client signatures or retry a blocked route. Results are draft-dev smoke, not a validated benchmark.

The Groq client uses the documented official Python SDK defaults. No custom client signature is used. The Colab checkout transport is patched before the network probe; the source commit alone does not describe this local transport change.


In [ ]:
from pathlib import Path
import subprocess, os, json, shutil
ROOT=Path('/content/agent-evals')
if not ROOT.exists():
    subprocess.run(['git','clone','https://github.com/yuvasree023/agent-evals.git',str(ROOT)],check=True)
print('Source commit:',subprocess.check_output(['git','rev-parse','HEAD'],cwd=ROOT,text=True).strip())
# Standard library only; no dependency installation needed.
subprocess.run(['python3','-m','unittest','discover','-s','tests'],cwd=ROOT,check=True)
subprocess.run(['python3','run.py','--validate'],cwd=ROOT,check=True)


In [ ]:
from google.colab import userdata
os.environ['GROQ_API_KEY']=userdata.get('GROQ_API_KEY')
# Gemini is optional for this Groq-agent run and is never printed.
try:
    os.environ['GEMINI_API_KEY']=userdata.get('GEMINI_API_KEY')
except userdata.SecretNotFoundError:
    pass
print('Groq secret loaded. No key is printed or saved to the notebook.')


In [ ]:
# Use Groq's documented official SDK with its default client identity.
# No browser identity, custom User-Agent, proxy, or signature override.
import sys
subprocess.run([sys.executable,'-m','pip','install','groq'],check=True)
import sys, importlib
sys.path.insert(0,str(ROOT))
path=ROOT/'providers.py'
source=path.read_text()
marker='    body = json.dumps(payload).encode()'
replacement='''    if provider == "groq":
        from groq import Groq, APIStatusError, APIConnectionError
        _pace(provider)
        try:
            client = Groq(api_key=_key(provider), max_retries=2, timeout=120)
            data = client.chat.completions.create(**payload)
            return 200, data.model_dump(), {}
        except APIStatusError as exc:
            if exc.response.headers.get("x-ratelimit-remaining-requests") == "0":
                raise DailyQuotaExhausted("groq daily request quota exhausted")
            raise ProviderError(f"groq SDK HTTP {exc.status_code}: {exc.response.text[:500]}")
        except APIConnectionError:
            raise ProviderError("groq SDK connection failed")
    body = json.dumps(payload).encode()'''
if marker not in source:
    raise RuntimeError('Provider transport marker missing')
# Ephemeral Colab checkout only. Provenance is printed, no secret values.
if 'from groq import Groq, APIStatusError' not in source:
    path.write_text(source.replace(marker,replacement))
import agent, providers
importlib.reload(providers)
from run import load_tasks
config=json.loads((ROOT/'config.json').read_text())
task=next(t for t in load_tasks() if t['split']=='dev')
answer, env, _=agent.run_task(task,'groq',config['default_models']['groq'],
    max_steps=config['max_steps'],cache_dir=str(ROOT/'.cache/sdk-probe'),use_cache=False)
env.db.close()
print('One dev task completed with Groq official SDK defaults.')


In [ ]:
# Run sequentially with free-tier pacing. A quota/error stop preserves checkpoints.
try:
    subprocess.run(['python3','run.py','--split','dev','--agent','groq',
        '--trials','4','--limit','35','--resume'],cwd=ROOT,check=True)
    subprocess.run(['python3','run.py','--split','dev','--agent','groq',
        '--limit','35','--flip-check','--resume'],cwd=ROOT,check=True)
finally:
    archive=shutil.make_archive('/content/m1-dev-results','zip',ROOT/'results/local')
    print('Results archive:',archive)


In [ ]:
from google.colab import files
files.download('/content/m1-dev-results.zip')


## Resuming

In the same runtime, rerun the trial cell after a verified quota reset. Completed trial checkpoints are reused; each task/trial has its own response cache. If the runtime disconnects, download the results first. A new runtime has no old cache automatically. Do not blindly repeat a 403/1010 access block. Keep full traces for human review; do not call smoke failures human labels.